In [28]:
import sys
sys.path.append("../")
import neo4j
import pandas as pd
from utils import simplify_to_centroid_if_small
from utils_neo4j import init, geofoxpois_insert_query
from geofox_client import get_geofox_client
from utils_geofox import get_pois, poisdf2rows
import pandas as pd
from datetime import datetime

In [2]:
client = get_geofox_client()

In [ ]:
def check_name(client,  maxList=1):
    sdName = {
        "type": "POI",
        "combinedName": "Hamburg",
    }

    endpoint = 'checkName' 
    
    request = {
    "language": "de",
    "version": 59,
    "tariffDetails": True,
    "maxList": maxList,
    "theName": sdName,
    }

    res = client.send(endpoint, request)

    return res

In [40]:
def get_route(client, start, dest, time=None, penalties=None):


    endpoint = 'getRoute' 

    # Beispiel für Abfahrtszeit
    # wenn None wird aktuelle Zeit verwendet
    # time = { 
    #         "date": "22.04.2025", "time": "18:30" 
    #     }
    
    # beispiel für penalty
    # penalties = [{
    #     "name": "DesiredType", "value": "u:-10"
    # }]
    
    request = {
    "language": "de",
    "version": 59,
    "tariffDetails": False,
    "start": start,
    "dest": dest,
    "time": time,   # Zeit im Format GTITime: abfahrtszeit wenn timeIsDeparture = True, ankunftszeit wenn timeIsDeparture = False
    "timeIsDeparture": True,
    "penalties": penalties,
    }

    res = client.send(endpoint, request)
    if 'realtimeAffected' in res and res['realtimeAffected']:
        schedules = res['realtimeSchedules']
    else:
        schedules = res['schedules']

    return schedules

In [41]:
start = { # Beh\u00f6rde f\u00fcr Stadtentwicklung und Wohnen
        "type": "COORDINATE",
        "coordinate": {
                "x": 10.004187,
                "y": 53.497465
            },
    }

dest = { # "Altonaer Segel-Club e.V."
        "type": "COORDINATE",
        "coordinate": {
                "x": 9.858205,
                "y": 53.537384
            },
    }

In [42]:
res2 = get_route(client, start, dest)
res2

[{'routeId': 0,
  'start': {'name': 'Gertrud-von-Thaden-Platz 1',
   'city': 'Hamburg',
   'combinedName': '(Nähe) Gertrud-von-Thaden-Platz 1',
   'type': 'COORDINATE',
   'coordinate': {'x': 10.004187, 'y': 53.497465}},
  'dest': {'name': 'Rüschweg 25',
   'city': 'Hamburg',
   'combinedName': '(Nähe) Rüschweg 25',
   'type': 'COORDINATE',
   'coordinate': {'x': 9.858205, 'y': 53.537384}},
  'time': 57,
  'footpathTime': 14,
  'plannedDepartureTime': '2025-04-22T16:43:00.000+0200',
  'realDepartureTime': '2025-04-22T16:43:00.000+0200',
  'plannedArrivalTime': '2025-04-22T17:40:00.000+0200',
  'realArrivalTime': '2025-04-22T17:42:00.000+0200',
  'tickets': [{'price': 3.9,
    'type': 'Einzelkarte HVV (EUR)',
    'level': 'Hamburg AB',
    'tariff': 'HVV'}],
  'scheduleElements': [{'from': {'name': 'Gertrud-von-Thaden-Platz 1',
     'city': 'Hamburg',
     'combinedName': '(Nähe) Gertrud-von-Thaden-Platz 1',
     'type': 'COORDINATE',
     'coordinate': {'x': 10.004187, 'y': 53.497465},

In [ ]:
time = { 
        "date": "22.04.2025", "time": "18:30" 
}

penalties = [{
    "name": "DesiredType", "value": "u:-10" # -> ubahn wird bevorzugt
}]

In [46]:
res2 = get_route(client, start, dest, None, penalties)
res2

[{'routeId': 0,
  'start': {'name': 'Gertrud-von-Thaden-Platz 1',
   'city': 'Hamburg',
   'combinedName': '(Nähe) Gertrud-von-Thaden-Platz 1',
   'type': 'COORDINATE',
   'coordinate': {'x': 10.004187, 'y': 53.497465}},
  'dest': {'name': 'Rüschweg 25',
   'city': 'Hamburg',
   'combinedName': '(Nähe) Rüschweg 25',
   'type': 'COORDINATE',
   'coordinate': {'x': 9.858205, 'y': 53.537384}},
  'time': 53,
  'footpathTime': 6,
  'plannedDepartureTime': '2025-04-22T17:27:00.000+0200',
  'realDepartureTime': '2025-04-22T17:27:00.000+0200',
  'plannedArrivalTime': '2025-04-22T18:20:00.000+0200',
  'realArrivalTime': '2025-04-22T18:20:00.000+0200',
  'tickets': [{'price': 3.9,
    'type': 'Einzelkarte HVV (EUR)',
    'level': 'Hamburg AB',
    'tariff': 'HVV'}],
  'scheduleElements': [{'from': {'name': 'Gertrud-von-Thaden-Platz 1',
     'city': 'Hamburg',
     'combinedName': '(Nähe) Gertrud-von-Thaden-Platz 1',
     'type': 'COORDINATE',
     'coordinate': {'x': 10.004187, 'y': 53.497465},
